# 05 - Model Explainability and Interpretation

This notebook demonstrates explainability techniques to interpret model predictions for dementia classification.

---

## Outline
- SHAP Values for Tabular Models
- Feature Importance Analysis
- SHAP Summary and Force Plots
- Grad-CAM for CNN Models (Optional)
- Model Decision Visualization
- Clinical Interpretation

---

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import shap
from sklearn.model_selection import train_test_split

# Add src to path
sys.path.append('../src')
from data_loading import load_clinical_data
from explainability import compute_shap_values, plot_shap_summary, plot_shap_force

# Display settings
pd.set_option('display.max_columns', 100)
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Initialize SHAP's JavaScript visualization in notebooks
shap.initjs()

## 1. Load Models and Data

In [ ]:
# Load pre-trained models
model_dir = '../models'
models = {}

try:
    with open(os.path.join(model_dir, 'random_forest.pkl'), 'rb') as f:
        models['Random Forest'] = pickle.load(f)
    
    with open(os.path.join(model_dir, 'gradient_boosting.pkl'), 'rb') as f:
        models['Gradient Boosting'] = pickle.load(f)
    
    with open(os.path.join(model_dir, 'preprocessor.pkl'), 'rb') as f:
        preprocessor = pickle.load(f)
    
    print("Loaded models successfully")
    
except FileNotFoundError as e:
    print(f"Error loading models: {e}")
    print("Please run notebook 02 first to train and save models.")
    models = None

In [ ]:
# Load and prepare test data
if models:
    try:
        clinical_path = '../data/raw/clinical.csv'
        df = load_clinical_data(clinical_path)
        
        # Define features and target
        numeric_features = ['Age', 'EDUC', 'MMSE', 'eTIV', 'nWBV', 'ASF']
        categorical_features = ['M/F']
        target_column = 'CDR'
        
        # Clean data
        df_clean = df.dropna(subset=[target_column])
        X = df_clean[[col for col in numeric_features + categorical_features if col in df_clean.columns]]
        y = df_clean[target_column]
        y_binary = (y > 0).astype(int)
        
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
        )
        
        # Preprocess
        X_test_processed = preprocessor.transform(X_test)
        feature_names = preprocessor.get_feature_names_out()
        X_test_processed_df = pd.DataFrame(
            X_test_processed, columns=feature_names, index=X_test.index
        )
        
        print(f"Test set shape: {X_test_processed_df.shape}")
        
    except Exception as e:
        print(f"Error loading data: {e}")
        X_test_processed_df = None

## 2. Feature Importance (Tree-based Models)

Analyze feature importance from Random Forest and Gradient Boosting models.

In [ ]:
# Plot feature importance
if models and X_test_processed_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, (name, model) in enumerate([('Random Forest', models['Random Forest']), 
                                           ('Gradient Boosting', models['Gradient Boosting'])]):
        # Get feature importance
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1][:15]  # Top 15 features
        
        # Plot
        ax = axes[idx]
        ax.barh(range(len(indices)), importances[indices], color='steelblue')
        ax.set_yticks(range(len(indices)))
        ax.set_yticklabels([feature_names[i] for i in indices])
        ax.set_xlabel('Feature Importance')
        ax.set_title(f'Top 15 Features - {name}')
        ax.invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    # Print top features
    for name, model in [('Random Forest', models['Random Forest']), 
                        ('Gradient Boosting', models['Gradient Boosting'])]:
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1][:10]
        print(f"\nTop 10 Features for {name}:")
        for i, idx in enumerate(indices, 1):
            print(f"{i}. {feature_names[idx]}: {importances[idx]:.4f}")

## 3. SHAP Values for Random Forest

Use SHAP to explain Random Forest predictions.

In [ ]:
# Compute SHAP values for Random Forest
if models and X_test_processed_df is not None:
    print("Computing SHAP values for Random Forest...")
    
    # Use a sample of the test set for faster computation
    sample_size = min(100, len(X_test_processed_df))
    X_sample = X_test_processed_df.sample(n=sample_size, random_state=42)
    
    rf_shap_values, rf_explainer = compute_shap_values(
        models['Random Forest'],
        X_sample,
        explainer_type="TreeExplainer"
    )
    
    print("SHAP values computed successfully")

## 4. SHAP Summary Plots

Visualize overall feature impact using SHAP summary plots.

In [ ]:
# SHAP summary plot (beeswarm)
if models and X_test_processed_df is not None:
    print("SHAP Summary Plot (Beeswarm):")
    # For binary classification, use the positive class (index 1)
    if isinstance(rf_shap_values, list):
        shap_values_plot = rf_shap_values[1]
    else:
        shap_values_plot = rf_shap_values
    
    shap.summary_plot(shap_values_plot, X_sample, plot_type="dot", show=False)
    plt.tight_layout()
    plt.show()

In [ ]:
# SHAP summary plot (bar)
if models and X_test_processed_df is not None:
    print("SHAP Summary Plot (Bar - Mean Absolute Impact):")
    shap.summary_plot(shap_values_plot, X_sample, plot_type="bar", show=False)
    plt.tight_layout()
    plt.show()

## 5. SHAP Force Plots for Individual Predictions

Examine how features contribute to individual predictions.

In [ ]:
# Force plot for a single prediction
if models and X_test_processed_df is not None:
    # Select a few interesting samples
    sample_indices = [0, 1, 2]  # First three samples
    
    for i in sample_indices:
        print(f"\nForce plot for sample {i}:")
        
        if isinstance(rf_shap_values, list):
            expected_value = rf_explainer.expected_value[1]
            shap_val = rf_shap_values[1][i]
        else:
            expected_value = rf_explainer.expected_value
            shap_val = rf_shap_values[i]
        
        # Create force plot
        shap.force_plot(
            expected_value,
            shap_val,
            X_sample.iloc[i],
            matplotlib=True,
            show=False
        )
        plt.tight_layout()
        plt.show()

## 6. SHAP Dependence Plots

Analyze the relationship between feature values and SHAP values.

In [ ]:
# Identify top features for dependence plots
if models and X_test_processed_df is not None:
    # Get top features by mean absolute SHAP value
    mean_abs_shap = np.abs(shap_values_plot).mean(axis=0)
    top_features_idx = np.argsort(mean_abs_shap)[::-1][:4]  # Top 4 features
    
    # Create dependence plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for i, feat_idx in enumerate(top_features_idx):
        plt.sca(axes[i])
        shap.dependence_plot(
            feat_idx,
            shap_values_plot,
            X_sample,
            show=False
        )
    
    plt.tight_layout()
    plt.show()

## 7. SHAP Waterfall Plot

Show cumulative impact of features for specific predictions.

In [ ]:
# Waterfall plot for individual predictions
if models and X_test_processed_df is not None:
    # Create waterfall plot for first sample
    sample_idx = 0
    
    if isinstance(rf_shap_values, list):
        expected_value = rf_explainer.expected_value[1]
        shap_val = rf_shap_values[1][sample_idx]
    else:
        expected_value = rf_explainer.expected_value
        shap_val = rf_shap_values[sample_idx]
    
    # Create Explanation object for waterfall plot
    explanation = shap.Explanation(
        values=shap_val,
        base_values=expected_value,
        data=X_sample.iloc[sample_idx].values,
        feature_names=X_sample.columns.tolist()
    )
    
    print(f"Waterfall plot for sample {sample_idx}:")
    shap.waterfall_plot(explanation, show=False)
    plt.tight_layout()
    plt.show()

## 8. SHAP Values for Gradient Boosting

Compare SHAP explanations across different models.

In [ ]:
# Compute SHAP values for Gradient Boosting
if models and X_test_processed_df is not None:
    print("Computing SHAP values for Gradient Boosting...")
    
    gbm_shap_values, gbm_explainer = compute_shap_values(
        models['Gradient Boosting'],
        X_sample,
        explainer_type="TreeExplainer"
    )
    
    print("SHAP values computed successfully")
    
    # SHAP summary plot for GBM
    print("\nSHAP Summary Plot for Gradient Boosting:")
    if isinstance(gbm_shap_values, list):
        gbm_shap_values_plot = gbm_shap_values[1]
    else:
        gbm_shap_values_plot = gbm_shap_values
    
    shap.summary_plot(gbm_shap_values_plot, X_sample, plot_type="dot", show=False)
    plt.tight_layout()
    plt.show()

## 9. Clinical Interpretation Summary

Summarize key clinical insights from explainability analysis.

In [ ]:
# Create summary of feature impacts
if models and X_test_processed_df is not None:
    print("Clinical Interpretation Summary:")
    print("="*60)
    
    # Calculate mean absolute SHAP values
    mean_abs_shap_rf = np.abs(shap_values_plot).mean(axis=0)
    mean_abs_shap_gbm = np.abs(gbm_shap_values_plot).mean(axis=0)
    
    # Create summary DataFrame
    summary_df = pd.DataFrame({
        'Feature': feature_names,
        'RF_SHAP_Impact': mean_abs_shap_rf,
        'GBM_SHAP_Impact': mean_abs_shap_gbm,
        'RF_Feature_Importance': models['Random Forest'].feature_importances_,
        'GBM_Feature_Importance': models['Gradient Boosting'].feature_importances_
    })
    
    # Calculate average importance across methods
    summary_df['Avg_Impact'] = (summary_df['RF_SHAP_Impact'] + summary_df['GBM_SHAP_Impact']) / 2
    summary_df = summary_df.sort_values('Avg_Impact', ascending=False)
    
    print("\nTop 10 Most Important Features (Averaged across models):")
    display(summary_df.head(10))
    
    # Save summary
    os.makedirs('../outputs', exist_ok=True)
    summary_df.to_csv('../outputs/feature_importance_summary.csv', index=False)
    print("\nFeature importance summary saved to ../outputs/feature_importance_summary.csv")

## 10. Save Explainability Visualizations

In [ ]:
# Save key visualizations
if models and X_test_processed_df is not None:
    os.makedirs('../outputs/figures', exist_ok=True)
    
    # SHAP summary plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values_plot, X_sample, plot_type="dot", show=False)
    plt.tight_layout()
    plt.savefig('../outputs/figures/shap_summary_rf.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # Feature importance bar plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values_plot, X_sample, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig('../outputs/figures/shap_bar_rf.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print("Visualizations saved to ../outputs/figures/")

## Summary

This notebook demonstrated:
- Feature importance analysis for tree-based models
- SHAP values computation and interpretation
- Various SHAP visualization techniques:
  - Summary plots (beeswarm and bar)
  - Force plots for individual predictions
  - Dependence plots showing feature relationships
  - Waterfall plots for cumulative impact
- Comparison of explainability across different models
- Clinical interpretation of model decisions

### Key Clinical Insights
The explainability analysis reveals which clinical and demographic features are most important for dementia prediction, providing interpretable insights for clinical decision-making.

### Next Steps
- Use these insights in the results and reporting notebook (06)
- Include explainability visualizations in publications
- Discuss clinical relevance of top features